# MOSAIC Boolean / Binary Value Detector
**Detection-only.** Read-only against source data. Every finding requires steward review before any action.

| Cell | What it does | Procedure §§ |
|---|---|---|
| 1 – Overview | This document | — |
| 2 – Rule Reference | Full rule definitions, standardization options, decision workflow | All |
| 3 – Config | Parameters / widgets | — |
| 4 – Constants | Tags, encoding maps, severity weights, standardization rules | All |
| 5 – Discovery | Table & column inventory from information_schema | — |
| 6 – Profiler | Aggregate stats for candidate boolean columns | §Canonical, §Validation |
| 7 – AI Classifier | Confirm boolean candidates; infer semantic direction & inverted risk | §Semantic, §Source mapping |
| 8 – Rule Engine | One function per rule group; detection logic | All |
| 9 – Write Results | Append findings to results Delta table | — |
| 10 – Summary | Blocking findings first, then ranked columns | §Validation, §Enforcement |

> **Hard constraint:** `UPDATE`, `MERGE`, `DELETE`, `CTAS` on source tables are never executed.
> **Scope:** All columns in the target catalog/schema are screened; only confirmed boolean/flag candidates receive full rule analysis.


# MOSAIC Boolean / Binary Value Detector — Rule Reference

This notebook detects encoding violations, mixed representations, unsafe NULL handling,
third-state overloading, and semantic documentation gaps in boolean/flag/binary columns.
It is **detection-only** — no source data is modified.

---

## How to read a finding

| Field | What it means |
|---|---|
| `classification` | The rule tag that fired (see below) |
| `rule_ref` | The procedure section that defines the rule |
| `detected_encoding` | The actual encoding found in this column (0/1, Y/N, True/False, mixed, etc.) |
| `semantic_direction` | What the value 1 means for this column — AI-inferred where possible |
| `inverted_risk` | `true` when the AI identifies a counter-intuitive direction (e.g. 1 = suppressed) |
| `affected_count` / `affected_pct` | How many rows triggered the rule |
| `standardization_rule` | JSON array of options; steward picks one and records `decided_action` |
| `decided_action` | Pre-populated when the procedure leaves no ambiguity; otherwise NULL until steward records choice |
| `needs_steward_review` | `false` only when `decided_action` is pre-populated from a single unambiguous option |

**Blocking findings** (`OUT_OF_RANGE_VALUE`, `MIXED_ENCODING`, `STRING_IN_SILVER_GOLD`, `BLANK_AS_BOOLEAN`)
appear first in the summary and may block Silver/Gold certification per the enforcement section.

---

## Decision workflow

1. Run the notebook — findings written with `decided_action = NULL`.
2. Steward reads `standardization_rule` (JSON array), picks the right `option_key`, records:
```sql
UPDATE <out_catalog>.<out_schema>.<out_table>
SET    decided_action = '<option_key>',
       decided_by     = '<steward name>'
WHERE  run_id = '<run_id>'
  AND  table_name = '<table>'
  AND  column_name = '<column>'
  AND  classification = '<tag>';
```
3. Engineer queries Section 5 of the summary for their work queue (`decided_action IS NOT NULL`).

---

## Strategy hint reference

| Hint | Strategy name | What it means | Typically seen on |
|---|---|---|---|
| **A** | Absent / missing | NULL is acceptable; document nullability | `UNEXPECTED_NULL` (nullable flag) |
| **B** | Not collected | NULL by design for this record type | `UNEXPECTED_NULL` (structural NULL) |
| **C** | Required / no nulls | Zero NULLs tolerated; quarantine missing | Flags that are mandatory indicators |
| **D** | Not applicable | Flag doesn't apply to this record type | `THIRD_STATE_STRING` where a state means N/A |

---

## Rule definitions

### `MIXED_ENCODING` ⚠ May block certification
**Procedure:** §Canonical — "Do not persist mixed encodings (Yes/No strings alongside 0/1) in Silver/Gold."

**What it checks:** A column contains both numeric boolean values (0/1) AND string boolean variants (Y/N, Yes/No, True/False, etc.) coexisting in the same column.

**Why it matters:** Mixed encodings break SUM/GROUP BY aggregations silently. `SUM(col)` on a mixed column returns only the numeric rows; string rows are ignored or cause type errors. This is a certification blocker in Silver/Gold.

**Standardization options:** Convert all values to canonical 0/1/NULL in Silver using the source mapping table.

---

### `NON_CANONICAL_ENCODING`
**Procedure:** §Canonical — "Persist binary fields as 0, 1, or NULL in curated tables."

**What it checks:** A column uses only string boolean variants (Y/N, Yes/No, True/False, T/F, etc.) with no numeric 0/1 present. Valid for Bronze; violation in Silver/Gold.

**Why it matters:** String booleans cannot be aggregated directly (`SUM(is_high_risk)` fails on 'Y'/'N'). Silver must store 0/1 as the canonical form per the cross-LOB consensus (Chad Serpan, JP Naan, Ryan Williams, Jonathan Walker — 2026-06-22 Architecture Sync).

**Standardization:** CASE WHEN UPPER(TRIM(col)) IN ('Y','YES','TRUE','T','1') THEN 1 WHEN UPPER(TRIM(col)) IN ('N','NO','FALSE','F','0') THEN 0 ELSE NULL END.

---

### `NATIVE_BOOLEAN_TYPE`
**Procedure:** §Canonical — "native BOOLEAN/BIT with documented semantics" is acceptable.

**What it checks:** Column is typed as BOOLEAN or BIT rather than TINYINT/INT with 0/1.

**Why it matters:** Acceptable per procedure, but requires a dictionary entry documenting true/false business meaning and source-value mapping. This is an informational flag — no transform needed, documentation required.

---

### `OUT_OF_RANGE_VALUE` ⚠ Blocks certification
**Procedure:** §Validation — "post-load values must be ∈ {0, 1, NULL}."

**What it checks:** A numeric column that was identified as a boolean flag contains values other than 0, 1, or NULL (e.g. 2, 9, -1, 99).

**Why it matters:** Out-of-range values corrupt every aggregation that assumes 0/1 semantics. They bypass type checks silently and cause silent metric errors downstream. Hard certification blocker.

**Standardization:** Quarantine rows with out-of-range values; investigate source system.

---

### `THIRD_STATE_STRING`
**Procedure:** §Two-state vs. unknown — "The agreed standard is a strict two-state 0/1 flag with NULL for unknown/not-applicable. If a field genuinely needs a distinct user-asserted 'Unknown' value, do not overload the boolean — model it as a categorical."

**What it checks:** A string boolean column has a third distinct non-null value beyond the expected true/false pair (e.g. Y/N/U, Yes/No/Unknown, 0/1/9).

**Why it matters:** The third state overloads the boolean contract. `SUM(col = 'Y')` gives wrong counts; analysts may treat the third state as NULL silently. Per procedure, this should either map to NULL (if it means unknown/not-applicable) or the column should be modelled as a categorical.

**Standardization options:** (a) Map third state to NULL if it means unknown/not-applicable. (b) Promote to categorical column and raise to governance. (c) Document as a governance-approved exception.

---

### `UNEXPECTED_NULL`
**Procedure:** §NULL handling — "Blank, Unknown, invalid → NULL or quarantine per Null and missing values."

**What it checks:** A confirmed boolean column has NULL values. The AI assesses whether NULLs are expected (nullable flag = Strategy A) or a data quality issue (missing required flag = Strategy C).

**Why it matters:** For flags that must always be set (e.g. `is_active`, `consent_given`), NULLs indicate a load failure or upstream data gap. For flags that apply only to some records, NULLs are correct and expected.

**Standardization:** Depends on AI classification — Strategy A (document nullability) or Strategy C (quarantine null rows).

---

### `BLANK_AS_BOOLEAN` ⚠ May block certification
**Procedure:** §Source mapping — "Blank → NULL or quarantine per Null and missing values."

**What it checks:** A string-typed boolean column contains empty string '' or whitespace-only values, which should have been mapped to NULL in Silver but weren't.

**Why it matters:** Empty strings pass `IS NOT NULL` checks silently and break aggregations that expect only Y/N/null semantics.

**Standardization:** Apply TRIM(); map '' to NULL in Silver transform.

---

### `UNDOCUMENTED_DIRECTION`
**Procedure:** §Downstream use — "Document semantic direction (1 = active vs 1 = suppressed) in the dictionary to prevent inverted metrics in reporting."

**What it checks:** AI cannot confidently infer what 1 means for this column from its name and sample data. Column name is too generic (e.g. `flag_1`, `status_bit`, `ind_b`) to determine direction without documentation.

**Why it matters:** Without documented direction, downstream analysts may invert the metric (e.g. `SUM(is_suppressed)` reported as engagement). This is a documentation gap, not a data error.

**Standardization:** Add dictionary entry documenting 1 = [meaning] and 0 = [meaning].

---

### `INVERTED_METRIC_RISK`
**Procedure:** §Downstream use — "Document semantic direction... to prevent inverted metrics."

**What it checks:** AI identifies column names where 1 = true creates counter-intuitive metrics — typically negative-state flags: `is_suppressed`, `is_excluded`, `is_deleted`, `is_blocked`, `is_inactive`, `is_rejected`.

**Why it matters:** `SUM(is_suppressed)` counts suppressed records, which reads as an engagement or volume metric to analysts unfamiliar with the direction. Ryan Williams flagged this pattern (Architecture Sync 2026-06-22): 0/1 aggregates cleanly but the direction must be documented.

**Standardization:** Add dictionary entry with explicit direction warning. Consider renaming to positive-state convention (`is_active` instead of `is_inactive`).

---

### `STRING_IN_SILVER_GOLD` ⚠ Blocks certification
**Procedure:** §Layer behavior — "Silver: store the canonical 0/1/NULL... Do not persist mixed encodings in Silver/Gold."

**What it checks:** A string-typed boolean column (storing Y/N, True/False, etc.) found in a layer the operator has declared as Silver or Gold.

**Why it matters:** The procedure requires 0/1 as Silver's canonical form. String boolean values in Silver/Gold are a direct policy violation requiring hotfix before republication.

**Standardization:** Apply the NON_CANONICAL_ENCODING transform immediately; re-publish Silver.

---

### `UNMAPPED_SOURCE_VALUE`
**Procedure:** §Source mapping — "Implement explicit transformation rules mapping all source variants."

**What it checks:** A boolean column contains values that don't match any known boolean variant (neither true-side: Y/Yes/True/T/1, nor false-side: N/No/False/F/0, nor NULL). AI attempts to classify these as potential boolean variants in other encodings or languages.

**Why it matters:** Unmapped values will be silently lost or produce errors when the source mapping is applied. They must be explicitly handled in the transformation rule.

**Standardization:** Document the unmapped value and its intended mapping (→ 1, → 0, or → NULL) in the rule repository.

---

## Extensibility — adding new rule modules

1. Add tag(s) to `TAGS` in **Cell 4** with procedure section reference
2. Add severity to `SEVERITY` in **Cell 4**
3. Add standardization options to `STANDARDIZATION_RULES` in **Cell 4**
4. Write a `_check_<topic>()` function in **Cell 8** and call it in the main loop
5. No other cells need changes


In [0]:
import re, json, uuid, hashlib
from datetime import datetime, date
from typing import Any, Dict, List, Optional, Tuple, Set
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    # Source
    "catalog":        _w("catalog",        "data_classification_source"),
    "schema":         _w("schema",         "binary_boolean_detector"),
    "table_filter":   _w("table_filter",   ""),
    "skip_views":     _w("skip_views",     "true").strip().lower() == "true",

    # Layer declaration -- affects severity of STRING_IN_SILVER_GOLD
    # Set to the layer you are scanning: Bronze, Silver, Gold, Unknown
    "layer":          _w("layer",          "Unknown"),

    # Sampling
    # sample_size: number of rows for value-level analysis (distinct value checks)
    "sample_size":    int(_w("sample_size",    10_000)),
    "seed":           int(_w("seed",           42)),

    # Candidate detection thresholds
    # max_bool_distinct: max distinct non-null values for a numeric column to be
    # considered a boolean candidate (should be <= 2 for strict 0/1; allow 3-4 for
    # columns with sentinel values like 9 or -1 mixed in)
    "max_bool_distinct": int(_w("max_bool_distinct", 4)),

    # Results
    "out_catalog":    _w("out_catalog",    "data_classification_target"),
    "out_schema":     _w("out_schema",     "data_classification_output"),
    "out_table":      _w("out_table",      "boolean_findings"),

    # AI
    "ai_model":       _w("ai_model",       "databricks-meta-llama-3-3-70b-instruct"),
    # enable_ai: when True, uses ai_query() for:
    #   (1) boolean column candidate confirmation on ambiguous STRING columns
    #   (2) semantic direction inference (what does 1 mean?)
    #   (3) inverted metric risk detection (1 = suppressed / excluded / deleted?)
    #   (4) unmapped source value classification (unknown encoding variants)
    "enable_ai":      _w("enable_ai",      "true").strip().lower() == "true",
}

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

print(f"Run      : {RUN_ID}")
print(f"Scope    : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer    : {CFG['layer']}  (affects STRING_IN_SILVER_GOLD severity)")
print(f"AI       : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off (heuristic-only)'}")
print(f"Results  : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# HOW TO ADD A NEW RULE MODULE:
#   1. Add tag to TAGS with procedure section reference
#   2. Add severity weight to SEVERITY
#   3. Add standardization options list to STANDARDIZATION_RULES
#   4. Write _check_<topic>() in Cell 8 and call it in the main loop
# ---------------------------------------------------------------------------

# ── Classification tags ──────────────────────────────────────────────────────
TAGS = {
    "MIXED_ENCODING":          "§Canonical",
    "NON_CANONICAL_ENCODING":  "§Canonical",
    "NATIVE_BOOLEAN_TYPE":     "§Canonical",
    "OUT_OF_RANGE_VALUE":      "§Validation",
    "THIRD_STATE_STRING":      "§Validation",
    "UNEXPECTED_NULL":         "§NULL",
    "BLANK_AS_BOOLEAN":        "§NULL",
    "UNDOCUMENTED_DIRECTION":  "§Semantic",
    "INVERTED_METRIC_RISK":    "§Semantic",
    "STRING_IN_SILVER_GOLD":   "§Layer",
    "UNMAPPED_SOURCE_VALUE":   "§SourceMapping",
}

SEVERITY: Dict[str, int] = {
    "OUT_OF_RANGE_VALUE":      10,   # hard certification blocker
    "MIXED_ENCODING":           9,   # certification blocker in Silver/Gold
    "STRING_IN_SILVER_GOLD":    9,   # direct policy violation in Silver/Gold
    "BLANK_AS_BOOLEAN":         7,   # silent aggregation break
    "NON_CANONICAL_ENCODING":   6,   # Silver requirement violated
    "THIRD_STATE_STRING":       5,   # boolean contract overloaded
    "UNEXPECTED_NULL":          5,   # depends on AI classification
    "INVERTED_METRIC_RISK":     4,   # misleading metric risk
    "UNMAPPED_SOURCE_VALUE":    4,   # transform gap
    "UNDOCUMENTED_DIRECTION":   2,   # documentation gap
    "NATIVE_BOOLEAN_TYPE":      1,   # informational
}

# ── Boolean variant sets ─────────────────────────────────────────────────────
# TRUE_VARIANTS / FALSE_VARIANTS: values that map to 1 / 0 per source mapping table
TRUE_VARIANTS:  Set[str] = {"y", "yes", "true", "t", "1", "si", "oui", "ja", "sim",
                             "on", "active", "enabled", "1.0"}
FALSE_VARIANTS: Set[str] = {"n", "no", "false", "f", "0", "non", "nein", "nao",
                             "off", "inactive", "disabled", "0.0"}
ALL_BOOL_STRINGS: Set[str] = TRUE_VARIANTS | FALSE_VARIANTS

# Numeric canonical values (the MOSAIC standard for Silver/Gold)
CANONICAL_NUMERIC: Set = {0, 1, 0.0, 1.0}

# Native boolean dtypes (acceptable but need documentation)
BOOLEAN_DTYPES: Set[str] = {"BOOLEAN", "BIT"}

# Numeric dtypes that could be boolean columns
NUMERIC_BOOL_DTYPES: Set[str] = {"TINYINT", "SMALLINT", "INT", "INTEGER", "BIGINT",
                                  "LONG", "BYTE"}

# String dtypes
STRING_DTYPES_PREFIX = ("STRING", "VARCHAR", "CHAR")

# ── Heuristic column-name patterns for boolean candidate detection ─────────────
# These are signals, not hard rules -- AI handles unseen conventions
RE_BOOL_NAME = re.compile(
    r"(^is_|^has_|^was_|^can_|^should_|^allow|^enable|^active|^flag|"
    r"_flag$|_ind$|_indicator$|_bit$|_bool$|_yn$|_tf$|"
    r"consent|verified|approved|confirmed|deleted|excluded|suppressed|"
    r"blocked|rejected|enrolled|eligible|opted|subscribed|valid|visible)",
    re.I
)

# Inverted-metric patterns: 1=negative-state is counter-intuitive
RE_INVERTED = re.compile(
    r"(is_suppressed|is_excluded|is_deleted|is_blocked|is_inactive|is_invalid|"
    r"is_rejected|is_banned|is_deactivated|is_suspended|is_hidden|is_archived|"
    r"is_cancelled|is_voided|is_errored|is_failed|is_missing|is_duplicate|"
    r"do_not_|opt_out|exclude_|suppress_|block_)",
    re.I
)

# ── Standardization rules ─────────────────────────────────────────────────────
# Each tag -> list of option dicts with option_key, label, sql, notes.
# decided_action is pre-populated only when len(options) == 1.

STANDARDIZATION_RULES: Dict[str, list] = {

    "MIXED_ENCODING": [
        {"option_key": "convert_all_to_canonical",
         "label":      "Convert all values to canonical 0/1/NULL (Silver transform)",
         "sql":        "CASE WHEN CAST(TRIM(col) AS STRING) IN ('1','y','yes','true','t') THEN 1 "
                       "     WHEN CAST(TRIM(col) AS STRING) IN ('0','n','no','false','f') THEN 0 "
                       "     ELSE NULL END",
         "notes":      "Apply this CASE expression in the Silver transform. "
                       "Mixed encodings must not persist in Silver/Gold (§Canonical). "
                       "Store the mapping in the rule repository linked to the pipeline."},
    ],

    "NON_CANONICAL_ENCODING": [
        {"option_key": "map_strings_to_01",
         "label":      "Map string boolean variants to 0/1/NULL (Silver transform)",
         "sql":        "CASE WHEN UPPER(TRIM(col)) IN ('Y','YES','TRUE','T','1') THEN 1 "
                       "     WHEN UPPER(TRIM(col)) IN ('N','NO','FALSE','F','0') THEN 0 "
                       "     ELSE NULL END",
         "notes":      "Silver canonical form is 0/1/NULL. Bronze preserves source string as-is. "
                       "Store the mapping in the rule repository. "
                       "Gold may relabel to TRUE/FALSE only on governance-approved exception (§Layer)."},
        {"option_key": "governance_exception_gold_only",
         "label":      "Document as governance-approved Gold relabel (TRUE/FALSE for report-facing)",
         "sql":        "-- Gold layer only: CAST(col_silver AS BOOLEAN) -- or CASE 1 THEN 'Yes' 0 THEN 'No' END",
         "notes":      "Only valid for Gold report-facing dimensions with board-approved exception. "
                       "Silver must still store 0/1. Never re-store string in Silver fact layer."},
    ],

    "NATIVE_BOOLEAN_TYPE": [
        {"option_key": "document_semantics",
         "label":      "Document true/false business meaning in data dictionary (no transform)",
         "sql":        "-- No transform needed. Add dictionary entry: true=[meaning], false=[meaning].",
         "notes":      "BOOLEAN/BIT columns are acceptable per procedure with documented semantics. "
                       "Required: dictionary entry with business meaning of true/false, "
                       "nullability, and source-value mapping (§Inventory)."},
    ],

    "OUT_OF_RANGE_VALUE": [
        {"option_key": "quarantine_out_of_range",
         "label":      "Quarantine rows with out-of-range values (blocks certification)",
         "sql":        "-- Route rows WHERE col NOT IN (0, 1) AND col IS NOT NULL to quarantine table.",
         "notes":      "Post-load values must be in {0, 1, NULL} (§Validation). "
                       "This is a hard certification blocker. Investigate source before republication."},
    ],

    "THIRD_STATE_STRING": [
        {"option_key": "map_third_to_null",
         "label":      "Map third state to NULL (it means unknown/not-applicable)",
         "sql":        "CASE WHEN UPPER(TRIM(col)) IN ('Y','YES','TRUE','T','1') THEN 1 "
                       "     WHEN UPPER(TRIM(col)) IN ('N','NO','FALSE','F','0') THEN 0 "
                       "     ELSE NULL END",
         "notes":      "Per procedure: NULL for unknown/not-applicable. "
                       "Do not overload boolean with a third string state (§Two-state vs. unknown)."},
        {"option_key": "promote_to_categorical",
         "label":      "Promote to categorical column and raise to governance",
         "sql":        "-- Model as a categorical column instead of boolean. Raise to governance board.",
         "notes":      "If the third state carries genuine business meaning (not just 'unknown'), "
                       "model it as a categorical per the procedure and raise to governance (§Two-state vs. unknown)."},
    ],

    "UNEXPECTED_NULL": [
        {"option_key": "document_as_nullable_strategy_a",
         "label":      "Document as nullable flag (Strategy A) -- NULL is expected",
         "sql":        "-- No transform. Add dictionary entry documenting nullability and its meaning.",
         "notes":      "NULL means the flag does not apply or was not recorded for this record. "
                       "Document in data dictionary: NULL = [meaning]."},
        {"option_key": "quarantine_null_rows_strategy_c",
         "label":      "Quarantine rows with NULL flag (Strategy C -- flag is required)",
         "sql":        "-- Route rows WHERE col IS NULL to quarantine table; exclude from Silver grain.",
         "notes":      "If this flag must always be set, NULL indicates a load gap. "
                       "Quarantine and investigate source."},
    ],

    "BLANK_AS_BOOLEAN": [
        {"option_key": "trim_map_blank_to_null",
         "label":      "Apply TRIM() and map empty string to NULL (Silver transform)",
         "sql":        "CASE WHEN TRIM(col) = '' THEN NULL "
                       "     WHEN UPPER(TRIM(col)) IN ('Y','YES','TRUE','T','1') THEN '1' "
                       "     WHEN UPPER(TRIM(col)) IN ('N','NO','FALSE','F','0') THEN '0' "
                       "     ELSE NULL END",
         "notes":      "Blank/empty string must map to NULL per source mapping table (§Source mapping). "
                       "Then apply NON_CANONICAL_ENCODING transform to get 0/1."},
    ],

    "UNDOCUMENTED_DIRECTION": [
        {"option_key": "add_dictionary_entry",
         "label":      "Add data dictionary entry documenting 1=[meaning] and 0=[meaning]",
         "sql":        "-- No transform. Dictionary entry required: 1=[true meaning], 0=[false meaning].",
         "notes":      "Semantic direction must be documented to prevent inverted metrics (§Downstream use). "
                       "Consider renaming to a self-documenting convention (e.g. is_active vs is_flag)."},
    ],

    "INVERTED_METRIC_RISK": [
        {"option_key": "document_direction_warning",
         "label":      "Document with explicit direction warning in data dictionary",
         "sql":        "-- No transform. Add dictionary warning: 1=[negative state] -- use SUM() with caution.",
         "notes":      "1 = a negative state (suppressed/excluded/deleted). "
                       "SUM() on this column counts negative outcomes, not positive ones. "
                       "Consider renaming to positive-state convention (§Downstream use)."},
        {"option_key": "rename_to_positive_convention",
         "label":      "Rename column to positive-state convention (governance change)",
         "sql":        "-- Rename: is_inactive -> is_active (invert logic). Requires governance approval.",
         "notes":      "Renaming requires a pipeline change, data dictionary update, and downstream "
                       "BI contract review. Raise to governance board before implementing."},
    ],

    "STRING_IN_SILVER_GOLD": [
        {"option_key": "apply_canonical_transform_immediately",
         "label":      "Apply canonical 0/1 transform immediately -- BLOCKS certification",
         "sql":        "CASE WHEN UPPER(TRIM(col)) IN ('Y','YES','TRUE','T','1') THEN 1 "
                       "     WHEN UPPER(TRIM(col)) IN ('N','NO','FALSE','F','0') THEN 0 "
                       "     ELSE NULL END",
         "notes":      "String boolean in Silver/Gold is a direct policy violation requiring hotfix "
                       "before republication (§Enforcement). Apply transform and regression-test."},
    ],

    "UNMAPPED_SOURCE_VALUE": [
        {"option_key": "map_to_true",
         "label":      "Map this value to 1 (confirmed true-side variant)",
         "sql":        "CASE WHEN UPPER(TRIM(col)) = '<value>' THEN 1 ELSE col END",
         "notes":      "Replace <value> with the actual unmapped value. "
                       "Store the mapping in the rule repository (§Source mapping)."},
        {"option_key": "map_to_false",
         "label":      "Map this value to 0 (confirmed false-side variant)",
         "sql":        "CASE WHEN UPPER(TRIM(col)) = '<value>' THEN 0 ELSE col END",
         "notes":      "Replace <value> with the actual unmapped value. Store in rule repository."},
        {"option_key": "map_to_null",
         "label":      "Map this value to NULL (confirmed blank/unknown variant)",
         "sql":        "CASE WHEN UPPER(TRIM(col)) = '<value>' THEN NULL ELSE col END",
         "notes":      "Replace <value> with the actual unmapped value. Store in rule repository."},
        {"option_key": "investigate_source",
         "label":      "Investigate source system -- value meaning unknown",
         "sql":        "-- No transform until source meaning confirmed.",
         "notes":      "Value does not map to any known boolean variant. "
                       "Investigate source system before adding to the mapping table."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} standardization entries")


In [0]:
# ---------------------------------------------------------------------------
# Reads Unity Catalog information_schema.
# Builds:
#   tables      : List[str]
#   table_cols  : Dict[str, List[(col, dtype)]]
# ---------------------------------------------------------------------------

def _live_cfg(name: str) -> str:
    try:
        value = dbutils.widgets.get(name)
        return value if value else str(CFG[name])
    except Exception:
        return str(CFG[name])

cat, sch = _live_cfg("catalog"), _live_cfg("schema")

if cat == "data_classification_source" and sch == "default":
    default_has_tables = spark.sql(f"""
        SELECT 1
        FROM system.information_schema.tables
        WHERE table_catalog = '{cat}'
          AND table_schema = '{sch}'
          AND table_type <> 'VIEW'
        LIMIT 1
    """).collect()
    if not default_has_tables:
        sch = "binary_boolean_detector"

view_clause = "AND table_type <> 'VIEW'" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name for r in spark.sql(f"""
        SELECT table_name
        FROM system.information_schema.tables
        WHERE table_catalog = '{cat}'
          AND table_schema = '{sch}' {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch} -- check catalog/schema/table_filter.")

tbl_in = ", ".join(f"'{t}'" for t in tables)
table_cols: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM   system.information_schema.columns
    WHERE  table_catalog = '{cat}' AND table_schema = '{sch}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    table_cols[r.table_name].append((r.column_name, r.data_type.upper()))

total_cols = sum(len(v) for v in table_cols.values())
print(f"Scope   : {cat}.{sch}")
print(f"Tables  : {len(tables)}")
print(f"Columns : {total_cols}")
print(f"Layer   : {CFG['layer']}")


In [0]:
# ---------------------------------------------------------------------------
# BOOLEAN CANDIDATE DETECTION (two-pass):
#
# Pass 1 -- one SQL per table: for every column collect:
#   null_count, distinct_count (from sample), dtype
#   For STRING columns: blank count + top-N distinct trimmed values
#   For NUMERIC columns: distinct values and counts
#   For BOOLEAN/BIT columns: null count only (already canonical type)
#
# Pass 2 -- classify candidates:
#   a) BOOLEAN/BIT dtype -> always candidate
#   b) NUMERIC (TINYINT/SMALLINT/INT/BIGINT): candidate if all non-null
#      distinct values subset of {0,1} (strict) or within max_bool_distinct
#      including sentinel-like values
#   c) STRING: candidate if all top-N distinct non-blank trimmed values
#      (lowercased) are in ALL_BOOL_STRINGS OR column name matches RE_BOOL_NAME
#      -> these go to AI classifier (Cell 7) for confirmation
#
# Output:
#   table_samples     : Dict[str, DataFrame]
#   bool_candidates   : Dict[str, Dict[str, dict]]
#     bool_candidates[tbl][col] = {
#         "dtype": str,
#         "null_count": int, "total": int,
#         "distinct_vals": list,          # top-N (value, count) pairs
#         "blank_count": int,             # string cols only
#         "candidate_reason": str,        # "dtype" / "numeric_range" / "string_values" / "name_heuristic"
#         "needs_ai_confirm": bool,       # True for name-heuristic and borderline string candidates
#     }
# ---------------------------------------------------------------------------

def _is_str(dt: str) -> bool:
    return any(dt.startswith(p) for p in STRING_DTYPES_PREFIX)

def _is_bool_dtype(dt: str) -> bool:
    return dt in BOOLEAN_DTYPES

def _is_numeric_bool_dtype(dt: str) -> bool:
    return dt in NUMERIC_BOOL_DTYPES

def mask(values: list, n: int = 5) -> list:
    return [str(v) for v in values if v is not None][:n]

def get_sample(tbl: str) -> DataFrame:
    fq = f"`{cat}`.`{sch}`.`{tbl}`"
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(f"{cat}.{sch}.{tbl}")
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )

def profile_table(tbl: str, cols: List[Tuple[str, str]]) -> Tuple[int, dict]:
    """One SQL query per table for all aggregate counts."""
    exprs = ["COUNT(*) AS __total__"]
    for col, dtype in cols:
        c    = f"`{col}`"
        safe = col.replace("`","").replace(" ","_")
        exprs.append(f"COUNT(*) - COUNT({c}) AS `__null__{safe}`")
        if _is_str(dtype):
            exprs.append(f"COUNT(CASE WHEN TRIM({c}) = '' THEN 1 END) AS `__blank__{safe}`")
    fq  = f"`{cat}`.`{sch}`.`{tbl}`"
    row = spark.sql(f"SELECT {', '.join(exprs)} FROM {fq}").collect()[0].asDict()
    total = row["__total__"]
    col_stats = {}
    for col, dtype in cols:
        safe = col.replace("`","").replace(" ","_")
        col_stats[col] = {
            "dtype":       dtype,
            "null_count":  row.get(f"__null__{safe}", 0) or 0,
            "blank_count": row.get(f"__blank__{safe}", 0) or 0,
            "total":       total,
        }
    return total, col_stats


def get_distinct_vals(tbl: str, col: str, dtype: str, sample_df: DataFrame) -> List[Tuple]:
    """Get top-N distinct trimmed values with counts from sample."""
    if _is_str(dtype):
        return [
            (r["tv"], r["count"]) for r in
            sample_df
            .select(F.trim(F.col(f"`{col}`")).alias("tv"))
            .filter(F.col("tv").isNotNull())
            .groupBy("tv").count()
            .orderBy(F.col("count").desc())
            .limit(30)
            .collect()
        ]
    else:
        return [
            (r["v"], r["count"]) for r in
            sample_df
            .select(F.col(f"`{col}`").alias("v"))
            .filter(F.col("v").isNotNull())
            .groupBy("v").count()
            .orderBy(F.col("count").desc())
            .limit(20)
            .collect()
        ]


def classify_candidate(col: str, dtype: str, stats: dict, distinct_vals: List[Tuple]) -> Optional[dict]:
    """
    Returns a candidate dict if the column looks like a boolean, else None.
    candidate_reason in: dtype / numeric_range / string_values / name_heuristic
    needs_ai_confirm=True means AI should verify before running rules.
    """
    vals_set = {str(v).strip().lower() for v, _ in distinct_vals}

    # a) Native boolean dtype -- always candidate, no AI needed
    if _is_bool_dtype(dtype):
        return {**stats, "distinct_vals": distinct_vals,
                "candidate_reason": "dtype", "needs_ai_confirm": False}

    # b) Numeric dtype -- candidate if all distinct values in {0,1} or near
    if _is_numeric_bool_dtype(dtype):
        numeric_vals = {v for v, _ in distinct_vals if v is not None}
        non_canonical = numeric_vals - CANONICAL_NUMERIC
        if not numeric_vals:
            return None
        if not non_canonical:
            return {**stats, "distinct_vals": distinct_vals,
                    "candidate_reason": "numeric_range", "needs_ai_confirm": False}
        # Borderline: few non-canonical values -- possible sentinel mixed in
        if len(numeric_vals) <= CFG["max_bool_distinct"] and len(non_canonical) <= 2:
            return {**stats, "distinct_vals": distinct_vals,
                    "candidate_reason": "numeric_range", "needs_ai_confirm": True}
        return None

    # c) String dtype
    if _is_str(dtype):
        non_blank = {v for v in vals_set if v}
        if not non_blank:
            return None
        # All values in known boolean strings -> strong candidate
        if non_blank and non_blank.issubset(ALL_BOOL_STRINGS):
            return {**stats, "distinct_vals": distinct_vals,
                    "candidate_reason": "string_values", "needs_ai_confirm": False}
        # Column name matches boolean pattern but values are ambiguous -> send to AI
        if RE_BOOL_NAME.search(col):
            return {**stats, "distinct_vals": distinct_vals,
                    "candidate_reason": "name_heuristic", "needs_ai_confirm": True}
        return None

    return None


# ── Run profiler ──────────────────────────────────────────────────────────────
table_samples:   Dict[str, DataFrame] = {}
bool_candidates: Dict[str, Dict[str, dict]] = defaultdict(dict)
total_screened   = 0
total_candidates = 0

for tbl in tables:
    cols = table_cols[tbl]
    total, col_stats = profile_table(tbl, cols)
    if total == 0:
        print(f"[SKIP] {tbl} -- empty table")
        continue

    sample_df = get_sample(tbl)
    table_samples[tbl] = sample_df

    for col, dtype in cols:
        total_screened += 1
        dist = get_distinct_vals(tbl, col, dtype, sample_df)
        candidate = classify_candidate(col, dtype, col_stats[col], dist)
        if candidate:
            bool_candidates[tbl][col] = candidate
            total_candidates += 1

print(f"Profiler done.")
print(f"  Screened : {total_screened} columns")
print(f"  Candidates: {total_candidates} boolean/flag columns")
for tbl, cols in bool_candidates.items():
    needs_ai = sum(1 for c in cols.values() if c["needs_ai_confirm"])
    print(f"  {tbl}: {len(cols)} candidate(s), {needs_ai} need AI confirmation")


In [0]:
# ---------------------------------------------------------------------------
# AI BOOLEAN CLASSIFIER -- three responsibilities:
#
# 1. CANDIDATE CONFIRMATION: For columns where needs_ai_confirm=True
#    (name-heuristic or borderline numeric), confirm whether it is genuinely
#    a boolean/flag. One ai_query() per TABLE (all ambiguous columns batched).
#
# 2. SEMANTIC DIRECTION: For all confirmed boolean candidates, infer what
#    1 means (semantic_direction) and flag inverted metric risk.
#    One ai_query() per TABLE (all confirmed boolean columns batched).
#
# 3. UNMAPPED VALUE CLASSIFICATION: For string boolean columns with values
#    not in ALL_BOOL_STRINGS, classify each unmapped value.
#    One ai_query() per column (called from Cell 8 rule engine).
#
# Output:
#   bool_candidates updated with:
#     "confirmed": bool           -- False removes from candidates
#     "semantic_direction": str   -- e.g. "1 = active, 0 = inactive"
#     "inverted_risk": bool       -- True if 1 = negative state
#     "semantic_source": str      -- ai / heuristic / both
#     "semantic_confidence": str  -- high / medium / low
# ---------------------------------------------------------------------------

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)


def _heuristic_direction(col: str) -> Optional[str]:
    """Best-effort heuristic semantic direction from column name."""
    c = col.lower()
    if any(x in c for x in ["active","enable","valid","verified","approved","eligible",
                              "enrolled","subscribed","confirmed","paid","completed"]):
        return f"1 = {c.replace('is_','').replace('_',' ')}, 0 = not"
    if any(x in c for x in ["suppress","exclude","delete","block","inactive","invalid",
                              "reject","ban","cancel","void","error","fail","missing",
                              "duplicate","opt_out","do_not"]):
        return f"1 = {c.replace('is_','').replace('_',' ')} (negative state), 0 = not"
    return None


def confirm_and_classify_table(tbl: str) -> None:
    """
    Run AI confirmation + direction inference for all candidates in one table.
    Modifies bool_candidates[tbl] in place.
    """
    cands = bool_candidates.get(tbl, {})
    if not cands:
        return

    needs_confirm = {col: info for col, info in cands.items() if info["needs_ai_confirm"]}
    confirmed_cols = {col for col, info in cands.items() if not info["needs_ai_confirm"]}

    # ── Step 1: Confirm ambiguous candidates ─────────────────────────────────
    if needs_confirm and CFG["enable_ai"]:
        payload = json.dumps([{
            "col":     col,
            "dtype":   info["dtype"],
            "samples": [str(v) for v, _ in info["distinct_vals"][:10]],
            "reason":  info["candidate_reason"],
            "null_pct": round(info["null_count"] / max(info["total"], 1) * 100, 1),
        } for col, info in needs_confirm.items()], default=str)

        prompt = (
            "Data classifier. Reply ONLY with a JSON array -- no prose, no markdown. "
            f"Table: {tbl}. For each column, determine if it is a boolean/flag column. "
            "A boolean/flag column stores exactly two states (true/false, yes/no, active/inactive, etc.) "
            "possibly with NULL for unknown. Any naming convention or language is valid. "
            f"Columns: {payload}. "
            'Return: [{"col":"<n>","is_boolean":true|false,"confidence":"high|medium|low","reason":"<1 sentence>"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                col = item.get("col", "")
                if col in needs_confirm:
                    if item.get("is_boolean", False):
                        confirmed_cols.add(col)
                        bool_candidates[tbl][col]["confirmed"] = True
                    else:
                        bool_candidates[tbl][col]["confirmed"] = False
        except Exception as e:
            print(f"  [WARN] AI candidate confirmation failed for {tbl}: {e}")
            # On failure, trust name-heuristic candidates, reject borderline numeric
            for col, info in needs_confirm.items():
                bool_candidates[tbl][col]["confirmed"] = (info["candidate_reason"] == "name_heuristic")
                if bool_candidates[tbl][col]["confirmed"]:
                    confirmed_cols.add(col)
    else:
        # AI off: trust name-heuristic, accept all dtype/string_values candidates
        for col, info in needs_confirm.items():
            keep = info["candidate_reason"] in ("name_heuristic", "string_values")
            bool_candidates[tbl][col]["confirmed"] = keep
            if keep:
                confirmed_cols.add(col)
        for col in [c for c, i in cands.items() if not i["needs_ai_confirm"]]:
            bool_candidates[tbl][col]["confirmed"] = True

    # Remove rejected candidates
    for col in list(bool_candidates[tbl].keys()):
        if not bool_candidates[tbl][col].get("confirmed", True):
            del bool_candidates[tbl][col]
            if col in confirmed_cols:
                confirmed_cols.discard(col)

    if not confirmed_cols:
        return

    # ── Step 2: Semantic direction + inverted risk for all confirmed columns ──
    # Apply heuristic first as baseline
    for col in confirmed_cols:
        h_dir = _heuristic_direction(col)
        bool_candidates[tbl][col]["semantic_direction"]   = h_dir
        bool_candidates[tbl][col]["inverted_risk"]        = RE_INVERTED.search(col) is not None
        bool_candidates[tbl][col]["semantic_source"]      = "heuristic"
        bool_candidates[tbl][col]["semantic_confidence"]  = "medium" if h_dir else "low"

    if CFG["enable_ai"]:
        payload = json.dumps([{
            "col":     col,
            "dtype":   bool_candidates[tbl][col]["dtype"],
            "samples": [str(v) for v, _ in bool_candidates[tbl][col]["distinct_vals"][:10]],
            "null_pct": round(bool_candidates[tbl][col]["null_count"] /
                              max(bool_candidates[tbl][col]["total"], 1) * 100, 1),
        } for col in confirmed_cols], default=str)

        prompt = (
            "Data governance advisor. Reply ONLY with a JSON array -- no prose, no markdown. "
            f"Table: {tbl}. For each confirmed boolean/flag column: "
            "(1) infer semantic_direction: a short phrase like '1 = active, 0 = inactive'. "
            "(2) set inverted_risk=true ONLY if 1 represents a NEGATIVE or undesirable state. "
            "NEGATIVE states (inverted_risk=true): suppressed, excluded, deleted, blocked, inactive, "
            "invalid, rejected, cancelled, banned, failed, missing, duplicate, disabled, deactivated, suspended. "
            "POSITIVE states (inverted_risk=false): active, enabled, valid, verified, approved, "
            "confirmed, enrolled, eligible, subscribed, consented, opted-in, visible, published. "
            "CRITICAL: is_active=positive, is_enabled=positive, valid_indicator=positive. "
            "is_inactive=negative, is_disabled=negative, is_invalid=negative. "
            "(3) confidence: high if column name makes direction obvious, medium if inferred from context, "
            "low if ambiguous. "
            f"Columns: {payload}. "
            'Return: [{"col":"<n>","semantic_direction":"<phrase>","inverted_risk":true|false,'
            '"confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                col = item.get("col", "")
                if col in confirmed_cols and col in bool_candidates[tbl]:
                    ai_dir  = item.get("semantic_direction", "")
                    ai_inv  = item.get("inverted_risk", False)
                    ai_conf = item.get("confidence", "low")
                    h_dir   = bool_candidates[tbl][col].get("semantic_direction")
                    # Guard: if AI says inverted_risk=True but column name matches
                    # a clear positive-state word and RE_INVERTED does not match,
                    # the AI overcalled -- reset to False to prevent false positives.
                    import re as _re
                    _pos = _re.compile(
                        r"(^|_)(active|enabled|valid|verified|approved|confirmed|"
                        r"enrolled|eligible|subscribed|consented|opted_in|"
                        r"complete|successful|paid|visible|published)(\b|$|_)", _re.I)
                    if ai_inv and _pos.search(col) and not RE_INVERTED.search(col):
                        ai_inv = False
                    bool_candidates[tbl][col].update({
                        "semantic_direction":  ai_dir or h_dir,
                        "inverted_risk":       ai_inv,
                        "semantic_source":     "both" if h_dir else "ai",
                        "semantic_confidence": ai_conf,
                    })
        except Exception as e:
            print(f"  [WARN] AI semantic direction failed for {tbl}: {e}")


# ── Run classifier for all tables ─────────────────────────────────────────────
for tbl in list(bool_candidates.keys()):
    confirm_and_classify_table(tbl)
    remaining = len(bool_candidates[tbl])
    print(f"  ok {tbl}: {remaining} confirmed boolean candidate(s)")

total_confirmed = sum(len(v) for v in bool_candidates.values())
print(f"\nAI classification done. {total_confirmed} confirmed boolean column(s) across {len(bool_candidates)} table(s).")


In [0]:
# ---------------------------------------------------------------------------
# HOW TO ADD A NEW RULE:
#   1. Write _check_<topic>(tbl, col, info, sample_df) -> list
#   2. Add one call in the main loop
#   3. Add tag + severity + standardization_rule in Cell 4
# ---------------------------------------------------------------------------
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

def _finding(tbl, col, dtype, info, tag, count, total, samples, action,
             hint=None, confidence="high",
             std_options=None, auto_decided_action=None) -> dict:
    pct      = round(count / total * 100, 4) if total else 0.0
    options  = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    decided  = (auto_decided_action
                if auto_decided_action is not None and len(options) == 1
                else None)
    needs_sr = decided is None
    return {
        "run_id":                  RUN_ID,
        "run_ts":                  RUN_TS,
        "catalog":                 cat,
        "schema":                  sch,
        "table_name":              tbl,
        "column_name":             col,
        "data_type":               dtype,
        "layer":                   CFG["layer"],
        "detected_encoding":       info.get("detected_encoding", "unknown"),
        "semantic_direction":      info.get("semantic_direction"),
        "inverted_risk":           bool(info.get("inverted_risk", False)),
        "semantic_source":         info.get("semantic_source", "heuristic"),
        "rule_ref":                TAGS.get(tag, "§?"),
        "classification":          tag,
        "affected_count":          int(count),
        "affected_pct":            float(pct),
        "total_rows":              int(total),
        "sample_values":           json.dumps(samples, default=str),
        "recommended_action":      action,
        "dictionary_strategy_hint": hint,
        "standardization_rule":    json.dumps(options, ensure_ascii=False),
        "confidence":              confidence,
        "needs_steward_review":    needs_sr,
        "decided_action":          decided,
        "decided_by":              None,
    }


# ── Detect encoding type from distinct values ─────────────────────────────────
def _detect_encoding(distinct_vals: List[Tuple]) -> str:
    """Classify what encoding this column uses from its distinct non-null values."""
    vals_lower = {str(v).strip().lower() for v, _ in distinct_vals if v is not None and str(v).strip()}
    has_numeric_bool = bool(vals_lower & {"0", "1", "0.0", "1.0"})
    has_string_bool  = bool(vals_lower & ALL_BOOL_STRINGS - {"0", "1", "0.0", "1.0"})
    has_true_bool    = any(str(v).strip().lower() in {"true","false"} for v, _ in distinct_vals)

    if not vals_lower:
        return "null_only"
    if has_numeric_bool and has_string_bool:
        return "mixed"
    if has_numeric_bool and not has_string_bool:
        return "0/1"
    if has_true_bool:
        return "True/False"
    if vals_lower & {"y","n","yes","no"}:
        return "Y/N"
    if vals_lower.issubset(ALL_BOOL_STRINGS):
        return "string_variants"
    return "unknown"


# ── §Canonical: encoding checks ───────────────────────────────────────────────
def _check_encoding(tbl, col, info, sample_df) -> list:
    dtype   = info["dtype"]
    total   = info["total"]
    dist    = info["distinct_vals"]
    enc     = _detect_encoding(dist)
    info["detected_encoding"] = enc
    findings = []

    # NATIVE_BOOLEAN_TYPE (informational)
    if _is_bool_dtype(dtype):
        findings.append(_finding(tbl, col, dtype, info, "NATIVE_BOOLEAN_TYPE",
            total - info["null_count"], total,
            mask([v for v, _ in dist[:5]]),
            "BOOLEAN/BIT column -- add dictionary entry with true/false business meaning, "
            "nullability, and source-value mapping (§Inventory).",
            confidence="high",
            auto_decided_action="document_semantics",
        ))
        return findings  # native type needs no further encoding checks

    # MIXED_ENCODING
    if enc == "mixed":
        findings.append(_finding(tbl, col, dtype, info, "MIXED_ENCODING",
            total - info["null_count"], total,
            mask([str(v) for v, _ in dist[:10]]),
            "Column mixes numeric 0/1 and string boolean variants. "
            "Mixed encodings must not persist in Silver/Gold (§Canonical, §Enforcement).",
            confidence="high",
            auto_decided_action="convert_all_to_canonical",
        ))
        return findings  # MIXED subsumes NON_CANONICAL

    # STRING_IN_SILVER_GOLD (layer-specific severity)
    if _is_str(dtype) and enc in ("Y/N", "True/False", "string_variants"):
        layer = CFG["layer"].lower()
        if layer in ("silver", "gold"):
            findings.append(_finding(tbl, col, dtype, info, "STRING_IN_SILVER_GOLD",
                total - info["null_count"], total,
                mask([str(v) for v, _ in dist[:5]]),
                f"String boolean encoding in {CFG['layer']} layer is a direct policy violation. "
                "Hotfix required before republication (§Layer, §Enforcement).",
                confidence="high",
                auto_decided_action="apply_canonical_transform_immediately",
            ))
        else:
            # Not Silver/Gold -- flag as NON_CANONICAL_ENCODING
            findings.append(_finding(tbl, col, dtype, info, "NON_CANONICAL_ENCODING",
                total - info["null_count"], total,
                mask([str(v) for v, _ in dist[:5]]),
                "String boolean encoding detected. Silver canonical form is 0/1/NULL. "
                "Apply source mapping in the Silver transform (§Canonical).",
            ))

    return findings


# ── §Validation: out-of-range and third state ─────────────────────────────────
def _check_values(tbl, col, info, sample_df) -> list:
    dtype    = info["dtype"]
    total    = info["total"]
    dist     = info["distinct_vals"]
    findings = []

    if _is_bool_dtype(dtype):
        return []  # BOOLEAN type enforces range at storage level

    if _is_numeric_bool_dtype(dtype) or info.get("candidate_reason") == "numeric_range":
        # Check for values outside {0, 1}
        out_of_range = [(v, c) for v, c in dist
                        if v is not None and v not in CANONICAL_NUMERIC]
        if out_of_range:
            oor_count = sum(c for _, c in out_of_range)
            findings.append(_finding(tbl, col, dtype, info, "OUT_OF_RANGE_VALUE",
                oor_count, total,
                mask([str(v) for v, _ in out_of_range]),
                f"Values outside {{0, 1, NULL}} detected: {[str(v) for v, _ in out_of_range[:5]]}. "
                "Post-load values must be in {0, 1, NULL} (§Validation). "
                "Certification blocker.",
                confidence="high",
                auto_decided_action="quarantine_out_of_range",
            ))

    if _is_str(dtype):
        # THIRD_STATE_STRING: distinct non-blank non-boolean values present
        vals_lower = [(str(v).strip().lower(), c) for v, c in dist
                      if v is not None and str(v).strip()]
        bool_side  = [(v, c) for v, c in vals_lower if v in ALL_BOOL_STRINGS]
        other_side = [(v, c) for v, c in vals_lower if v not in ALL_BOOL_STRINGS]
        if bool_side and other_side:
            other_count = sum(c for _, c in other_side)
            findings.append(_finding(tbl, col, dtype, info, "THIRD_STATE_STRING",
                other_count, total,
                mask([v for v, _ in other_side]),
                f"Third state value(s) {[v for v, _ in other_side[:3]]} found alongside boolean pair. "
                "Do not overload boolean -- map to NULL or promote to categorical (§Two-state vs. unknown).",
                confidence="high",
            ))

    return findings


# ── §NULL: null and blank handling ────────────────────────────────────────────
def _check_nulls(tbl, col, info, sample_df) -> list:
    dtype      = info["dtype"]
    total      = info["total"]
    null_count = info["null_count"]
    blank_count = info.get("blank_count", 0)
    findings   = []

    # BLANK_AS_BOOLEAN: empty/whitespace in string boolean column
    if _is_str(dtype) and blank_count > 0:
        findings.append(_finding(tbl, col, dtype, info, "BLANK_AS_BOOLEAN",
            blank_count, total,
            ["''"],
            "Empty/whitespace string in a boolean column. "
            "Blank must map to NULL per source mapping table (§Source mapping, §NULL).",
            confidence="high",
            auto_decided_action="trim_map_blank_to_null",
        ))

    # UNEXPECTED_NULL: nulls present -- AI assessed which strategy applies
    if null_count > 0:
        sem_conf = info.get("semantic_confidence", "low")
        sem_dir  = info.get("semantic_direction", "")
        # Heuristic: if column name has 'required', 'mandatory', 'consent', 'opt'
        # -> likely Strategy C. Otherwise Strategy A.
        req_pattern = re.compile(r"(required|mandatory|consent|opt_in|must|obligat)", re.I)
        hint_strategy = "C" if req_pattern.search(col) else "A"
        findings.append(_finding(tbl, col, dtype, info, "UNEXPECTED_NULL",
            null_count, total, [],
            f"Boolean column has {round(null_count/total*100,1)}% NULL values. "
            f"Determine if NULL is expected (nullable flag, Strategy A) or a data gap (Strategy C). "
            f"Semantic direction: {sem_dir or 'not determined'}.",
            hint=hint_strategy, confidence=sem_conf or "medium",
        ))

    return findings


# ── §Semantic: direction and inverted risk ────────────────────────────────────
def _check_semantics(tbl, col, info) -> list:
    findings   = []
    sem_dir    = info.get("semantic_direction")
    inv_risk   = info.get("inverted_risk", False)
    sem_conf   = info.get("semantic_confidence", "low")
    dtype      = info["dtype"]
    total      = info["total"]

    if not sem_dir or sem_conf == "low":
        findings.append(_finding(tbl, col, dtype, info, "UNDOCUMENTED_DIRECTION",
            total - info["null_count"], total, [],
            "Cannot confidently determine what 1 means for this column from name and data. "
            "Add a data dictionary entry documenting 1=[meaning] and 0=[meaning] (§Downstream use).",
            confidence="high",
            auto_decided_action="add_dictionary_entry",
        ))
    elif inv_risk:
        findings.append(_finding(tbl, col, dtype, info, "INVERTED_METRIC_RISK",
            total - info["null_count"], total, [],
            f"1 = a negative state ({sem_dir}). "
            f"SUM() on this column counts negative outcomes, not positive ones. "
            "Document direction warning in the data dictionary (§Downstream use).",
            confidence=sem_conf,
        ))

    return findings


# ── §Source mapping: unmapped values (AI-assisted) ────────────────────────────
def _check_unmapped(tbl, col, info) -> list:
    """Detect values not in any known boolean variant set."""
    if not _is_str(info["dtype"]):
        return []

    dist = info["distinct_vals"]
    unmapped = [
        (v, c) for v, c in dist
        if v is not None
        and str(v).strip() != ""
        and str(v).strip().lower() not in ALL_BOOL_STRINGS
    ]
    if not unmapped:
        return []

    dtype = info["dtype"]
    total = info["total"]

    # AI classification of unmapped values
    if CFG["enable_ai"]:
        payload = json.dumps(
            [{"value": str(v), "count": c, "pct": round(c/max(sum(x for _,x in dist),1)*100,1)}
             for v, c in unmapped[:20]], default=str
        )
        prompt = (
            "Data classifier. Reply ONLY with a JSON array -- no prose, no markdown. "
            f"Table: {tbl}, Column: {col}. These values appear in a boolean/flag column "
            "but don't match any known boolean encoding (Y/N/Yes/No/True/False/0/1 etc.). "
            "For each value, determine if it is: "
            "true_variant (maps to 1 -- e.g. 'Si', 'Oui', 'Ja', 'Active', 'On', 'Enabled', '✓'), "
            "false_variant (maps to 0 -- e.g. 'Non', 'Nein', 'Inactive', 'Off', 'Disabled', '✗'), "
            "null_variant (maps to NULL -- e.g. blank, 'Unknown', 'N/A', 'Pending'), "
            "or unknown (cannot determine). "
            f"Values: {payload}. "
            'Return: [{"value":"<v>","mapping":"true_variant|false_variant|null_variant|unknown",'
            '"confidence":"high|medium|low"}]'
        )
        try:
            parsed = json.loads(_ai_query(prompt))
            mapping_map = {item["value"]: item for item in parsed}
        except Exception as e:
            print(f"  [WARN] AI unmapped value classification failed for {tbl}.{col}: {e}")
            mapping_map = {}
    else:
        mapping_map = {}

    findings = []
    mapping_opts = {
        "true_variant":  "map_to_true",
        "false_variant": "map_to_false",
        "null_variant":  "map_to_null",
        "unknown":       "investigate_source",
    }
    for v, c in unmapped:
        ai_info  = mapping_map.get(str(v), {})
        ai_map   = ai_info.get("mapping", "unknown")
        ai_conf  = ai_info.get("confidence", "low") if mapping_map else "heuristic_only"
        opt_key  = mapping_opts.get(ai_map, "investigate_source")
        findings.append(_finding(tbl, col, dtype, info, "UNMAPPED_SOURCE_VALUE",
            c, total, [str(v)],
            f"Value '{v}' does not match any known boolean encoding. "
            f"AI classification: {ai_map} (confidence: {ai_conf}). "
            "Add explicit mapping to the rule repository (§Source mapping).",
            confidence=ai_conf,
            auto_decided_action=opt_key if ai_map != "unknown" else None,
        ))
    return findings


# ── Main loop ─────────────────────────────────────────────────────────────────
all_findings: List[dict] = []

for tbl, cols in bool_candidates.items():
    sample_df = table_samples.get(tbl)
    if sample_df is None:
        continue
    tbl_count = 0
    for col, info in cols.items():
        col_findings = (
            _check_encoding(tbl, col, info, sample_df)
            + _check_values(tbl, col, info, sample_df)
            + _check_nulls(tbl, col, info, sample_df)
            + _check_semantics(tbl, col, info)
            + _check_unmapped(tbl, col, info)
        )
        all_findings.extend(col_findings)
        tbl_count += len(col_findings)
    print(f"  ok {tbl}: {tbl_count} finding(s)")

print(f"\nRule engine done. {len(all_findings)} total finding(s).")


In [0]:
# The only write in this notebook. Source tables are never modified.
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

SCHEMA = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("run_ts",                   StringType(),  True),
    StructField("catalog",                  StringType(),  True),
    StructField("schema",                   StringType(),  True),
    StructField("table_name",               StringType(),  True),
    StructField("column_name",              StringType(),  True),
    StructField("data_type",                StringType(),  True),
    StructField("layer",                    StringType(),  True),
    StructField("detected_encoding",        StringType(),  True),
    StructField("semantic_direction",       StringType(),  True),
    StructField("inverted_risk",            BooleanType(), True),
    StructField("semantic_source",          StringType(),  True),
    StructField("rule_ref",                 StringType(),  True),
    StructField("classification",           StringType(),  True),
    StructField("affected_count",           LongType(),    True),
    StructField("affected_pct",             DoubleType(),  True),
    StructField("total_rows",               LongType(),    True),
    StructField("sample_values",            StringType(),  True),
    StructField("recommended_action",       StringType(),  True),
    StructField("dictionary_strategy_hint", StringType(),  True),
    StructField("standardization_rule",     StringType(),  True),
    StructField("confidence",               StringType(),  True),
    StructField("needs_steward_review",     BooleanType(), True),
    StructField("decided_action",           StringType(),  True),
    StructField("decided_by",               StringType(),  True),
])

out_fq  = f"`{CFG['out_catalog']}`.`{CFG['out_schema']}`.`{CFG['out_table']}`"
out_tbl = f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG['out_catalog']}`.`{CFG['out_schema']}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
# ---------------------------------------------------------------------------
# Section 1 -- Blocking findings (read first)
# Section 2 -- Findings by classification
# Section 3 -- Findings by table
# Section 4 -- Inverted metric risks (documentation priority)
# Section 5 -- Engineer work queue (decided_action IS NOT NULL)
# ---------------------------------------------------------------------------

BLOCKING = {"OUT_OF_RANGE_VALUE", "MIXED_ENCODING", "STRING_IN_SILVER_GOLD", "BLANK_AS_BOOLEAN"}

if not all_findings:
    print("No boolean/flag columns found matching criteria. Run complete.")
else:
    fdf = findings_df

    # Section 1 -- Blocking
    block_df = fdf.filter(F.col("classification").isin(BLOCKING)) \
                  .orderBy("classification", F.col("affected_pct").desc())
    n_block = block_df.count()
    print("=" * 70)
    print(f"  BLOCKING FINDINGS: {n_block}  (may block Silver/Gold certification)")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","data_type","detected_encoding",
            "classification","rule_ref","affected_count","affected_pct",
            "recommended_action","standardization_rule","decided_action","decided_by"
        ))
    else:
        print("  None detected.")

    # Section 2 -- By classification
    print("\n" + "-" * 70)
    print("SECTION 2 -- Findings by classification")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.countDistinct("column_name").alias("columns"),
               F.sum("affected_count").alias("affected_rows"),
           ).orderBy(F.col("findings").desc())
    )

    # Section 3 -- By table
    print("\n" + "-" * 70)
    print("SECTION 3 -- Findings by table")
    print("-" * 70)
    display(
        fdf.groupBy("table_name")
           .agg(
               F.count("*").alias("total_findings"),
               F.sum(F.when(F.col("classification").isin(BLOCKING), 1).otherwise(0)).alias("blocking"),
               F.countDistinct("column_name").alias("boolean_columns"),
               F.sum(F.when(F.col("inverted_risk"), 1).otherwise(0)).alias("inverted_risks"),
           ).orderBy(F.col("blocking").desc(), F.col("total_findings").desc())
    )

    # Section 4 -- Inverted metric risks
    inv_df = fdf.filter(F.col("classification") == "INVERTED_METRIC_RISK")
    n_inv  = inv_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 -- Inverted metric risks ({n_inv})")
    print("  Columns where 1 = a negative state (suppressed/excluded/deleted/blocked)")
    print("  SUM() on these columns counts negative outcomes -- document and warn downstream users")
    print("-" * 70)
    if n_inv:
        display(inv_df.select(
            "table_name","column_name","semantic_direction","semantic_source",
            "recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None detected.")

    # Section 5 -- Engineer work queue
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 -- Engineer work queue  ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","classification","decided_action",
            "decided_by","standardization_rule"
        ).orderBy("table_name","column_name"))
    else:
        print("  No decisions recorded yet.")
        print("  Update decided_action in the findings table after steward review.")
        print("  Query: SELECT * FROM " + f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print("         WHERE run_id = '" + RUN_ID + "' AND decided_action IS NULL")
        print("         ORDER BY table_name, column_name;")

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Candidates: {total_confirmed} boolean column(s)")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  Inverted risks: {n_inv}")
    print(f"  Results: {out_fq}")
    print("=" * 70)
    print()
    print("Detection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
